# S16 — Transformer I

**Week 9 · Mon Oct 19, 2026 · Module 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s16_transformer_i.ipynb)

Every cell below is a worked example from the [S16 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s16/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s16.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s16.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
try:
    import torch  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
    import torch  # noqa: F401
print("environment ready")

## Positional encoding: giving order back


*Expected output starts with:* `no PE:  ||attn(shuffled x) - shuffle(attn(x))|| = 0.00000012`


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# Self-attention alone is permutation-equivariant: shuffle the input
# tokens and the outputs shuffle the same way -- word order is invisible.
# Adding positional encodings breaks the symmetry.

d_model, T = 32, 6

class SelfAttention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.q = nn.Linear(d, d); self.k = nn.Linear(d, d); self.v = nn.Linear(d, d)
    def forward(self, x):
        s = self.q(x) @ self.k(x).transpose(-2, -1) / math.sqrt(x.shape[-1])
        return F.softmax(s, dim=-1) @ self.v(x)

def sinusoidal_pe(T, d):
    pos = torch.arange(T, dtype=torch.float32).unsqueeze(1)
    i = torch.arange(0, d, 2, dtype=torch.float32)
    angles = pos / (10000 ** (i / d))
    pe = torch.zeros(T, d)
    pe[:, 0::2] = torch.sin(angles)
    pe[:, 1::2] = torch.cos(angles)
    return pe

attn = SelfAttention(d_model)
x = torch.randn(1, T, d_model)          # 6 "tokens"
perm = torch.tensor([3, 1, 5, 0, 2, 4]) # a shuffle of positions

# no positional encoding: output of shuffled input == shuffled output
y = attn(x)
y_perm = attn(x[:, perm, :])
print(f"no PE:  ||attn(shuffled x) - shuffle(attn(x))|| = "
      f"{(y_perm - y[:, perm, :]).abs().max().item():.8f}")

# with positional encoding: the symmetry is broken
pe = sinusoidal_pe(T, d_model).unsqueeze(0)
z = attn(x + pe)
z_perm = attn(x[:, perm, :] + pe)
print(f"with PE: ||attn(shuffled x+PE) - shuffle(attn(x+PE))|| = "
      f"{(z_perm - z[:, perm, :]).abs().max().item():.8f}")

# The sinusoidal table itself: each row is a unique "timestamp" vector.
pe_t = sinusoidal_pe(T, d_model)
print(f"\nPE row 0, first 6 dims: {[f'{v:.4f}' for v in pe_t[0, :6]]}")
print(f"PE row 3, first 6 dims: {[f'{v:.4f}' for v in pe_t[3, :6]]}")

## Three generations of positional encoding


*Expected output starts with:* `score q.k with RoPE applied, at various (query pos m, key pos n):`


In [ ]:
import numpy as np

np.random.seed(0)

# Rotary position embedding (RoPE). Instead of ADDING a position vector
# to the embedding, RoPE ROTATES each (even, odd) pair of query/key
# dimensions by an angle proportional to the token's position, with a
# different base frequency per pair. The payoff is exact relative
# position dependence: the dot product of a query at position m with a
# key at position n depends only on m - n, never on m and n themselves.

d = 8  # must be even

def rope(x, pos):
    i = np.arange(d // 2)
    theta = 10000.0 ** (-2.0 * i / d)      # one frequency per dimension pair
    ang = pos * theta
    cos, sin = np.cos(ang), np.sin(ang)
    x2 = x.reshape(d // 2, 2)
    out = np.empty_like(x2)
    out[:, 0] = x2[:, 0] * cos - x2[:, 1] * sin   # 2-D rotation of each pair
    out[:, 1] = x2[:, 0] * sin + x2[:, 1] * cos
    return out.reshape(d)

q = np.random.randn(d)
k = np.random.randn(d)

print("score q.k with RoPE applied, at various (query pos m, key pos n):")
for m, n in [(3, 1), (13, 11), (103, 101), (5, 1), (54, 50), (9, 1)]:
    s = rope(q, m) @ rope(k, n)
    print(f"  m = {m:>3}, n = {n:>3}  (m - n = {m - n}):  score = {s:.10f}")

# Contrast: ADDITIVE sinusoidal encodings do not have this property.
def sin_pe(pos):
    i = np.arange(d // 2)
    ang = pos * 10000.0 ** (-2.0 * i / d)
    pe = np.empty(d)
    pe[0::2] = np.sin(ang); pe[1::2] = np.cos(ang)
    return pe

print("\nscore (q + PE[m]).(k + PE[n]) with additive sinusoidal encodings:")
for m, n in [(3, 1), (13, 11), (103, 101)]:
    s = (q + sin_pe(m)) @ (k + sin_pe(n))
    print(f"  m = {m:>3}, n = {n:>3}  (m - n = {m - n}):  score = {s:.10f}")

## A minimal block in PyTorch, verified


*Expected output starts with:* `input shape  (2, 8, 64)`


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)   # Q, K, V in one matmul
        self.proj = nn.Linear(d_model, d_model)      # output projection W_O

    def forward(self, x, causal=True):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # (B, T, D) -> (B, n_heads, T, d_head)
        def split(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        q, k, v = split(q), split(k), split(v)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)  # (B, H, T, T)
        if causal:
            mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
            scores = scores.masked_fill(mask, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        out = attn @ v                                             # (B, H, T, d_head)
        out = out.transpose(1, 2).contiguous().view(B, T, D)       # concat heads
        return self.proj(out)

class TransformerBlock(nn.Module):
    """Pre-norm block: x + Attn(LN(x)), then x + FFN(LN(x))."""
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x, causal=True):
        x = x + self.attn(self.ln1(x), causal=causal)
        x = x + self.ffn(self.ln2(x))
        return x

B, T, d_model, n_heads, d_ff = 2, 8, 64, 4, 256
block = TransformerBlock(d_model, n_heads, d_ff)
x = torch.randn(B, T, d_model)
y = block(x)
n_params = sum(p.numel() for p in block.parameters())
print(f"input shape  {tuple(x.shape)}")
print(f"output shape {tuple(y.shape)}")
print(f"parameters   {n_params}")

# Causal-mask property: change the token at position 5; outputs at
# positions 0-4 must be bit-for-bit unchanged, positions 5-7 may change.
x2 = x.clone()
x2[:, 5, :] = torch.randn(B, d_model)
y2 = block(x2)
diff = (y2 - y).abs().amax(dim=(0, 2))  # max abs change per position
for t in range(T):
    print(f"position {t}: max |change| = {diff[t].item():.6f}")

# Same edit without the causal mask: everything changes.
y_nc, y2_nc = block(x, causal=False), block(x2, causal=False)
diff_nc = (y2_nc - y_nc).abs().amax(dim=(0, 2))
print(f"no mask, positions 0-4 max |change| = {diff_nc[:5].amax().item():.6f}")

## Counting a real model, by hand


*Expected output starts with:* `token embeddings        38,597,376`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Parameter accounting for a full GPT-2-small-shaped model, twice: once
# by hand with the formulas, once by instantiating real modules and
# counting. The two must agree exactly.

V, T_ctx, d, L, n_heads = 50257, 1024, 768, 12, 12
d_ff = 4 * d

# --- by hand ---
tok_emb = V * d                     # token embedding table
pos_emb = T_ctx * d                 # learned position embeddings
attn = (3 * d * d + 3 * d) + (d * d + d)      # QKV proj + output proj
ffn = (d * d_ff + d_ff) + (d_ff * d + d)      # two linear layers
ln = 2 * (2 * d)                    # two LayerNorms per block (scale + shift)
block = attn + ffn + ln
final_ln = 2 * d
head_untied = V * d                 # output projection to vocab logits

total_tied = tok_emb + pos_emb + L * block + final_ln          # head shares tok_emb
total_untied = total_tied + head_untied

print(f"token embeddings      {tok_emb:>12,}")
print(f"position embeddings   {pos_emb:>12,}")
print(f"one block: attention  {attn:>12,}")
print(f"one block: FFN        {ffn:>12,}")
print(f"one block: LayerNorms {ln:>12,}")
print(f"one block total       {block:>12,}   x {L} layers = {L * block:,}")
print(f"final LayerNorm       {final_ln:>12,}")
print(f"TOTAL (tied head)     {total_tied:>12,}")
print(f"TOTAL (untied head)   {total_untied:>12,}")
print(f"FFN share of block params: {ffn / block:.1%}")

# --- by instantiation ---
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.ffn = nn.Sequential(nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d))

class GPT2Small(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(V, d)
        self.pos = nn.Embedding(T_ctx, d)
        self.blocks = nn.ModuleList(Block() for _ in range(L))
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, V, bias=False)
        self.head.weight = self.tok.weight       # weight tying

model = GPT2Small()
counted = sum(p.numel() for p in {id(p): p for p in model.parameters()}.values())
print(f"\ninstantiated and counted (deduplicating tied weights): {counted:,}")
print(f"matches hand arithmetic: {counted == total_tied}")

## Try it yourself

1. Stack three `TransformerBlock`s and rerun the perturbation test. Does the "past is untouched" property survive depth? Why must it?
2. Compute the parameter count of the paper's base layer (`d_model = 512`, `n_heads = 8`, `d_ff = 2048`) by hand from the formulas above, then verify by instantiating the block and counting.
3. Convert the block to post-norm (`LayerNorm(x + sublayer(x))`) and compare the norm of the output of a 12-block stack (random weights, random input) against the pre-norm version. Which grows with depth, and what does that imply for gradients?
4. Replace the sinusoidal encoding with a learned `nn.Embedding(T, d_model)` of positions and rerun the permutation experiment. Does the symmetry still break before any training?
5. In the RoPE script, the relative-position identity was verified for dot products of *rotated* vectors. Check what rotation does to norms: print `||rope(q, m)||` for several `m`. What property of rotation matrices explains the result, and why does it matter that the encoding not change vector lengths?
6. Redo the parameter count for GPT-2 medium (`L = 24`, `d = 1024`, context and vocabulary unchanged). Before running, predict the embedding table's share of the total, and check how the FFN share of block parameters moves. Verify by instantiation.


---

Full discussion of everything above: [S16 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s16/).
